In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import os


**---Model---** matching our c++ architecture

In [3]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(784, 128)
        self.fc2=nn.Linear(128, 64)
        self.fc3=nn.Linear(64, 10)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x=self.fc3(x)
        return x

**Downloading Model**

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False)

model = MLP()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.51MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 97.2kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 941kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.43MB/s]


**Training**

In [5]:
print("Training...")
for epoch in range(5):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.view(-1, 784)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}/5  loss={total_loss/len(train_loader):.4f}")

Training...
  Epoch 1/5  loss=0.4078
  Epoch 2/5  loss=0.1642
  Epoch 3/5  loss=0.1159
  Epoch 4/5  loss=0.0860
  Epoch 5/5  loss=0.0691


**Evaluate**

In [6]:
model.eval()
correct = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.view(-1, 784)
        preds = model(imgs).argmax(dim=1)
        correct += (preds == labels).sum().item()
print(f"PyTorch accuracy: {correct}/{len(test_data)} = {correct/len(test_data)*100:.2f}%")


PyTorch accuracy: 9734/10000 = 97.34%


**Export weights as .npy**

In [7]:
os.makedirs('weights', exist_ok=True)
np.save('weights/fc1_w.npy', model.fc1.weight.detach().numpy())  
np.save('weights/fc1_b.npy', model.fc1.bias.detach().numpy())  
np.save('weights/fc2_w.npy', model.fc2.weight.detach().numpy())  
np.save('weights/fc2_b.npy', model.fc2.bias.detach().numpy())  
np.save('weights/fc3_w.npy', model.fc3.weight.detach().numpy())  
np.save('weights/fc3_b.npy', model.fc3.bias.detach().numpy())  

**Apply same normalization as training**

In [9]:
imgs_all  = test_data.data.numpy().astype(np.float32)[:1000]  # [1000, 28, 28]
labels_all = test_data.targets.numpy()[:1000]

imgs_all = (imgs_all / 255.0 - 0.1307) / 0.3081
imgs_all = imgs_all.reshape(1000, 784)  # flatten

np.save('weights/test_images.npy', imgs_all)   # [1000, 784]
np.save('weights/test_labels.npy', labels_all) # [1000]

print("Weights and test data exported to weights/")
print(f"  fc1_w: {model.fc1.weight.shape}")
print(f"  fc1_b: {model.fc1.bias.shape}")
print(f"  fc2_w: {model.fc2.weight.shape}")
print(f"  fc2_b: {model.fc2.bias.shape}")
print(f"  fc3_w: {model.fc3.weight.shape}")
print(f"  fc3_b: {model.fc3.bias.shape}")
print(f"  test_images: (1000, 784)")
print(f"  test_labels: (1000,)")

Weights and test data exported to weights/
  fc1_w: torch.Size([128, 784])
  fc1_b: torch.Size([128])
  fc2_w: torch.Size([64, 128])
  fc2_b: torch.Size([64])
  fc3_w: torch.Size([10, 64])
  fc3_b: torch.Size([10])
  test_images: (1000, 784)
  test_labels: (1000,)
